In [114]:
import pandas as pd
import numpy as np

In [115]:


arquivo     = "archive/yellow_tripdata_2015-01.csv"
arquivo1    = "archive/yellow_tripdata_2016-01.csv"
arquivo2    = "archive/yellow_tripdata_2016-02.csv"
arquivo3    = "archive/yellow_tripdata_2016-03.csv"

df  = pd.read_csv(arquivo, nrows=1000, low_memory=False)
df1 = pd.read_csv(arquivo1, nrows=1000, low_memory=False)
df2 = pd.read_csv(arquivo2, nrows=1000, low_memory=False)
df3 = pd.read_csv(arquivo3, nrows=1000, low_memory=False)

df["arquivo_origem"]    = arquivo
df1["arquivo_origem"]   = arquivo1
df2["arquivo_origem"]   = arquivo2
df3["arquivo_origem"]   = arquivo3

dfs = [df, df1, df2, df3]
for df in dfs:
    df.columns = df.columns.str.strip()
    df.rename(columns={"RatecodeID": "RateCodeID"}, inplace=True)


df_final = pd.concat(dfs, ignore_index=True)


In [116]:
# converter datas
df_final["tpep_pickup_datetime"] = pd.to_datetime(df_final["tpep_pickup_datetime"], errors="coerce")
df_final["tpep_dropoff_datetime"] = pd.to_datetime(df_final["tpep_dropoff_datetime"], errors="coerce")


df_final["ano_mes_pickup"] = df_final["tpep_pickup_datetime"].dt.strftime("%Y-%m")
df_final["ano_mes_esperado"] = df_final["arquivo_origem"].str.extract(r"(\d{4}-\d{2})")[0]

In [117]:
# Vamos cortar as datas para ajuste futuro 

df_final["pickup_date"]      = df_final["tpep_pickup_datetime"].dt.normalize()
df_final["pickup_year"]      = df_final["tpep_pickup_datetime"].dt.year
df_final["pickup_month"]     = df_final["tpep_pickup_datetime"].dt.month
df_final["pickup_day"]       = df_final["tpep_pickup_datetime"].dt.day
df_final["pickup_hour"]      = df_final["tpep_pickup_datetime"].dt.hour

df_final["dropoff_date"]     = df_final["tpep_dropoff_datetime"].dt.normalize()
df_final["dropoff_year"]     = df_final["tpep_dropoff_datetime"].dt.year
df_final["dropoff_month"]    = df_final["tpep_dropoff_datetime"].dt.month
df_final["dropoff_day"]      = df_final["tpep_dropoff_datetime"].dt.day
df_final["dropoff_hour"]     = df_final["tpep_dropoff_datetime"].dt.hour

In [118]:
# duração em minutos
df_final["trip_duration_min"] = (df_final["tpep_dropoff_datetime"] - df_final["tpep_pickup_datetime"] ).dt.total_seconds() / 60

# duração em horas
df_final["trip_duration_hr"] = df_final["trip_duration_min"] / 60 # aqui não corre risco porque estamos lidando com horas, e a divisão por zero ou negativa já será tratada na etapa de velocidade média

# velocidade média estimada
# Foi realizado tratamento por divisão por zero, substituindo por NaN e depois preenchendo com 0 para casos onde a duração é zero ou negativa.

df_final["trip_duration_hr_safe"] = df_final["trip_duration_hr"].replace(0, np.nan)
df_final["avg_speed_mph"] = df_final["trip_distance"] / df_final["trip_duration_hr_safe"]
df_final["avg_speed_mph"] = df_final["avg_speed_mph"].fillna(0)

In [119]:
# máscaras de duplicados    
mask_duplicado = df_final.duplicated(
    subset=[
        "VendorID",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "fare_amount",
        "total_amount"
    ],
    keep=False
)

# calcular valor total a partir dos componentes
df_final["valor_calculado"] = (
    df_final["fare_amount"].fillna(0) +
    df_final["extra"].fillna(0) +
    df_final["mta_tax"].fillna(0) +
    df_final["tip_amount"].fillna(0) +
    df_final["tolls_amount"].fillna(0) +
    df_final["improvement_surcharge"].fillna(0)
)

mask_total_inconsistente = (
    (df_final["total_amount"] - df_final["valor_calculado"]).abs() > 0.01
)


In [120]:

# máscaras de invalidez
mask_distancia_invalida = (df_final["trip_distance"] <= 0) | (df_final["trip_distance"] > 100)

mask_passageiros_invalidos = (df_final["passenger_count"] <= 0) | (df_final["passenger_count"] > 6)

mask_tarifa_invalida = (df_final["fare_amount"] <= 0) | (df_final["fare_amount"] > 500)

mask_duracao_invalida = (df_final["trip_duration_min"] <= 0) | (df_final["trip_duration_min"] > 300)

mask_velocidade_invalida = (
    (df_final["avg_speed_mph"] <= 0) |
    (df_final["avg_speed_mph"] > 85)
)

In [121]:
# Categorias validas

vendors_validos         = {1, 2}
payment_types_validos   = {1, 2, 3, 4, 5, 6}
store_fwd_validos       = {"Y", "N"}
rate_codes_validos      = {1, 2, 3, 4, 5, 6}

mask_vendor_invalido        = ~df_final["VendorID"].isin(vendors_validos)
mask_payment_type_invalido  = ~df_final["payment_type"].isin(payment_types_validos)
mask_store_fwd_invalido     = ~df_final["store_and_fwd_flag"].isin(store_fwd_validos)
mask_ratecode_invalido      = ~df_final["RateCodeID"].isin(rate_codes_validos)

mask_ordem_tempo_invalida = (
    df_final["tpep_dropoff_datetime"] < df_final["tpep_pickup_datetime"]
)

mask_data_nula = (
    df_final["tpep_pickup_datetime"].isna() |
    df_final["tpep_dropoff_datetime"].isna()
)

In [113]:
df_final.head(2)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RateCodeID,store_and_fwd_flag,dropoff_longitude,...,flag_vendor_invalido,flag_payment_type_invalido,flag_store_fwd_invalido,flag_ratecode_invalido,flag_ordem_tempo_invalida,flag_data_nula,flag_fora_mes,flag_pickup_coord_invalida,flag_dropoff_coord_invalida,flag_registro_invalido
0,2,2015-01-15 19:05:39,2015-01-15 19:23:42,1,1.59,-73.993896,40.750111,1,N,-73.974785,...,False,False,False,False,False,False,2015-01,False,False,True
1,1,2015-01-10 20:33:38,2015-01-10 20:53:28,1,3.30,-74.001648,40.724243,1,N,-73.994415,...,False,False,False,False,False,False,2015-01,False,False,True


In [122]:
mask_fora_mes = (
    df_final["ano_mes_esperado"] != df_final["ano_mes_pickup"]
)
mask_pickup_coord_invalida = (
    (df_final["pickup_longitude"] < -74.30) | (df_final["pickup_longitude"] > -73.65) |
    (df_final["pickup_latitude"] < 40.45) | (df_final["pickup_latitude"] > 40.95)
)

mask_dropoff_coord_invalida = (
    (df_final["dropoff_longitude"] < -74.30) | (df_final["dropoff_longitude"] > -73.65) |
    (df_final["dropoff_latitude"] < 40.45) | (df_final["dropoff_latitude"] > 40.95)
)

In [123]:
# máscara única
mask_invalido = (
    mask_duplicado |
    mask_total_inconsistente |
    mask_distancia_invalida |
    mask_passageiros_invalidos |
    mask_tarifa_invalida |
    mask_duracao_invalida |
    mask_velocidade_invalida |
    mask_vendor_invalido |
    mask_payment_type_invalido |
    mask_store_fwd_invalido |
    mask_ratecode_invalido |
    mask_ordem_tempo_invalida |
    mask_data_nula |
    mask_fora_mes |
    mask_pickup_coord_invalida |
    mask_dropoff_coord_invalida
)

df_final["flag_duplicado"] = mask_duplicado
df_final["flag_total_inconsistente"] = mask_total_inconsistente
df_final["flag_distancia_invalida"] = mask_distancia_invalida
df_final["flag_passageiros_invalidos"] = mask_passageiros_invalidos
df_final["flag_tarifa_invalida"] = mask_tarifa_invalida
df_final["flag_duracao_invalida"] = mask_duracao_invalida
df_final["flag_velocidade_invalida"] = mask_velocidade_invalida
df_final["flag_vendor_invalido"] = mask_vendor_invalido
df_final["flag_payment_type_invalido"] = mask_payment_type_invalido
df_final["flag_store_fwd_invalido"] = mask_store_fwd_invalido
df_final["flag_ratecode_invalido"] = mask_ratecode_invalido
df_final["flag_ordem_tempo_invalida"] = mask_ordem_tempo_invalida
df_final["flag_data_nula"] = mask_data_nula
df_final["flag_fora_mes"] = mask_fora_mes
df_final["flag_pickup_coord_invalida"] = mask_pickup_coord_invalida
df_final["flag_dropoff_coord_invalida"] = mask_dropoff_coord_invalida
df_final["flag_registro_invalido"] = mask_invalido

In [124]:
flag_cols = [
    "flag_duplicado",
    "flag_total_inconsistente",
    "flag_distancia_invalida",
    "flag_passageiros_invalidos",
    "flag_tarifa_invalida",
    "flag_duracao_invalida",
    "flag_velocidade_invalida",
    "flag_vendor_invalido",
    "flag_payment_type_invalido",
    "flag_store_fwd_invalido",
    "flag_ratecode_invalido",
    "flag_ordem_tempo_invalida",
    "flag_data_nula",
    "flag_fora_mes",
    "flag_pickup_coord_invalida",
    "flag_dropoff_coord_invalida"
]

df_final["qtd_regras_invalidas"] = df_final[flag_cols].astype(int).sum(axis=1)

In [125]:
score_qualidade_lote = round(
    (1 - df_final["flag_registro_invalido"].mean()) * 100,
    2
)

max_erros_possiveis = len(flag_cols) * len(df_final)

score_qualidade_lote_regras = round(
    (1 - (df_final["qtd_regras_invalidas"].sum() / max_erros_possiveis)) * 100,
    2
)

# separar válidos e inválidos
df_validos = df_final[~df_final["flag_registro_invalido"]].copy()
df_invalidos = df_final[df_final["flag_registro_invalido"]].copy()


print("Score do lote (por registros válidos):", score_qualidade_lote)
print("Score do lote (por regras violadas):", score_qualidade_lote_regras)
print("Total registros:", len(df_final))
print("Válidos:", len(df_validos))
print("Inválidos:", len(df_invalidos))

Score do lote (por registros válidos): 95.62
Score do lote (por regras violadas): 99.55
Total registros: 4000
Válidos: 3825
Inválidos: 175


In [127]:
df_load_final = df_validos.copy()

In [128]:
df_load_final["pickup_date"] = df_load_final["pickup_date"].dt.strftime("%Y-%m-%d")
df_load_final["dropoff_date"] = df_load_final["dropoff_date"].dt.strftime("%Y-%m-%d")

In [129]:
df_load_final["pickup_datetime"] = df_load_final["tpep_pickup_datetime"].dt.strftime("%Y-%m-%d %H:%M:%S")
df_load_final["dropoff_datetime"] = df_load_final["tpep_dropoff_datetime"].dt.strftime("%Y-%m-%d %H:%M:%S")

In [130]:
df_load_final = df_load_final.drop(
    columns=[
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime"
    ],
    errors="ignore"
)

In [131]:
print(df_load_final[[
    "pickup_date",
    "dropoff_date",
    "pickup_datetime",
    "dropoff_datetime"
]].head())

print(df_load_final.dtypes)

  pickup_date dropoff_date      pickup_datetime     dropoff_datetime
0  2015-01-15   2015-01-15  2015-01-15 19:05:39  2015-01-15 19:23:42
1  2015-01-10   2015-01-10  2015-01-10 20:33:38  2015-01-10 20:53:28
2  2015-01-10   2015-01-10  2015-01-10 20:33:38  2015-01-10 20:43:41
3  2015-01-10   2015-01-10  2015-01-10 20:33:39  2015-01-10 20:35:31
4  2015-01-10   2015-01-10  2015-01-10 20:33:39  2015-01-10 20:52:58
VendorID                         int64
passenger_count                  int64
trip_distance                  float64
pickup_longitude               float64
pickup_latitude                float64
RateCodeID                       int64
store_and_fwd_flag              object
dropoff_longitude              float64
dropoff_latitude               float64
payment_type                     int64
fare_amount                    float64
extra                          float64
mta_tax                        float64
tip_amount                     float64
tolls_amount                   float64
i